# House Prices Prediction - First Linear Regression Model

As as recently started PhD student, I'm self teaching a lot of the machine learning fundementals and after finishing the Ng Machine Learning and Deep Learning coursera courses. I'm now moving onto applying those courses to real life projects. This is my first attempt at making a linear regression model. 

I went through data analysis, handling null values, and used different regression models including linear, lasso and ridge. I also used gradient descent and selecting a smaller number of features. I compared the model performance and chose the best performing one. Currently ranked 3,082/6,594 participants.


Key learning points looking back on this:
- LLMs are the most useful resource for checking your code
- I found this Notebook a useful addition to check that my data analysis was correct: https://www.kaggle.com/code/arezoodahesh/house-price-prediction-linear-nonlinear-models





### Load the data

start just loading the data

In [68]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
/kaggle/input/home-data-for-ml-course/sample_submission.csv
/kaggle/input/home-data-for-ml-course/sample_submission.csv.gz
/kaggle/input/home-data-for-ml-course/train.csv.gz
/kaggle/input/home-data-for-ml-course/data_description.txt
/kaggle/input/home-data-for-ml-course/test.csv.gz
/kaggle/input/home-data-for-ml-course/train.csv
/kaggle/input/home-data-for-ml-course/test.csv


In [69]:
# Load the data, and separate the target
test_file_path = '/kaggle/input/home-data-for-ml-course/test.csv'
train_file_path = '/kaggle/input/home-data-for-ml-course/train.csv'

# Load the CSV file into a DataFrame
test_data = pd.read_csv(test_file_path)

# Display the first few rows
print(test_data.head())


# Load the CSV file into a DataFrame
train_data = pd.read_csv(train_file_path)

# Display the first few rows
print(train_data.head())


# Separate target variable (SalePrice)
y = train_data["SalePrice"]

     Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0  1461          20       RH         80.0    11622   Pave   NaN      Reg   
1  1462          20       RL         81.0    14267   Pave   NaN      IR1   
2  1463          60       RL         74.0    13830   Pave   NaN      IR1   
3  1464          60       RL         78.0     9978   Pave   NaN      IR1   
4  1465         120       RL         43.0     5005   Pave   NaN      IR1   

  LandContour Utilities  ... ScreenPorch PoolArea PoolQC  Fence MiscFeature  \
0         Lvl    AllPub  ...         120        0    NaN  MnPrv         NaN   
1         Lvl    AllPub  ...           0        0    NaN    NaN        Gar2   
2         Lvl    AllPub  ...           0        0    NaN  MnPrv         NaN   
3         Lvl    AllPub  ...           0        0    NaN    NaN         NaN   
4         HLS    AllPub  ...         144        0    NaN    NaN         NaN   

  MiscVal MoSold  YrSold  SaleType  SaleCondition  
0       0      6

In [ ]:
train_data.head()

In [ ]:
test_data.head()

In [ ]:
column_names = train_data.columns.tolist()
print(column_names)

In [ ]:
# check sample sumibssion format

sample_submission_path = '/kaggle/input/home-data-for-ml-course/sample_submission.csv'


# Load the CSV file into a DataFrame
sample_submission = pd.read_csv(sample_submission_path)

sample_submission.head()

# Display the data



Best Features for Scatter Plots:

#### Size-Related Features:
GrLivArea (Above-ground living area in square feet) ✅
TotalBsmtSF (Total basement area) ✅
1stFlrSF (First floor square footage) ✅
2ndFlrSF (Second floor square footage) ✅
LotArea (Lot size in square feet) ✅


#### Quality & Condition Indicators:
OverallQual (Overall material and finish quality, 1-10 scale) ✅
OverallCond (Overall condition of the house, 1-10 scale) ✅


#### Year Built & Renovation:
YearBuilt (Year the house was built) ✅
YearRemodAdd (Year the house was remodeled) ✅


#### Garage Features:
GarageArea (Garage size in square feet) ✅
GarageCars (Number of cars the garage can fit) ✅


#### Bathroom & Room Features:
TotRmsAbvGrd (Total rooms above ground) ✅
FullBath (Number of full bathrooms) ✅


In [ ]:
# List of selected features
features = ['GrLivArea', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 
            'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 
            'YearRemodAdd', 'GarageArea', 'GarageCars', 'TotRmsAbvGrd', 'FullBath']

target = 'SalePrice'

# Create scatter plots for selected features
plt.figure(figsize=(12, 12))
for i, feature in enumerate(features):
    plt.subplot(4, 4, i + 1)
    plt.scatter(train_data[feature], train_data[target], alpha=0.5)
    plt.xlabel(feature)
    plt.ylabel(target)
    plt.title(f"{feature} vs {target}")

plt.tight_layout()
plt.show()

In [ ]:
train_data.describe()

In [ ]:

# Histogram of house prices (SalePrice)
plt.figure(figsize=(10, 6))
plt.hist(train_data['SalePrice'], bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Sale Price')
plt.ylabel('Frequency')
plt.title('Distribution of House Prices')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Select only numeric columns
numeric_train_data = train_data.select_dtypes(include=['number'])

# Heatmap of correlation matrix
# Increase figure size for better visibility
plt.figure(figsize=(18, 12))

# Create heatmap with adjusted font size for better readability
sns.heatmap(numeric_train_data.corr(), cmap="coolwarm", annot=True, fmt=".2f", linewidths=0.5, annot_kws={"size": 10})

plt.title("Feature Correlation Heatmap", fontsize=16)
plt.xticks(fontsize=12, rotation=45)
plt.yticks(fontsize=12, rotation=0)
plt.show()

In [ ]:
correlation=(numeric_train_data.corr()['SalePrice'])
del correlation['SalePrice']
correlation.sort_values(ascending=True).plot(kind='barh',color = "darkgreen", 
                                             figsize = (100,90), fontsize=50)
plt.title("correlation", fontsize=50)
plt.axvline(x = 0.6, ymin = 0, ymax = 1,
            linewidth = 5, linestyle ="--",
            color ='#da1934');
plt.axvline(x = -0.6, ymin = 0, ymax = 1,
            linewidth = 5, linestyle ="--",
            color ='#da1934');

In [ ]:

# Compute correlation with SalePrice
correlation_with_target = numeric_train_data.corr()[['SalePrice']].sort_values(by='SalePrice', ascending=False)

# Increase figure size for better visibility
plt.figure(figsize=(8, 10))

# Create heatmap focused on SalePrice correlation
sns.heatmap(correlation_with_target, cmap="coolwarm", annot=True, fmt=".2f", linewidths=0.5, annot_kws={"size": 12})

plt.title("Correlation of Features with Sale Price", fontsize=16)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12, rotation=0)
plt.show()


In [ ]:
# Select non-numeric columns
categorical_features = train_data.select_dtypes(exclude=['number'])

# Create bar plots for each categorical feature
fig, axes = plt.subplots(nrows=len(categorical_features.columns)//3 + 1, ncols=3, figsize=(18, 30))
axes = axes.flatten()

for i, col in enumerate(categorical_features.columns):
    category_counts = train_data[col].value_counts()
    axes[i].bar(category_counts.index.astype(str), category_counts.values)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Count")
    axes[i].set_title(f"Distribution of {col}")
    axes[i].tick_params(axis='x', rotation=45)

# Hide unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
train_data.describe().T

In [ ]:
# see the type of data each column is

print("train_data: {}".format(train_data.info()))
print("\n************************************************************\n")
print("test_data: {}".format(test_data.info()))

# Data pre-processing

Now the data is converted into numerical with one-hot encoding and exammined, time to develop a linear regression model.

## Train test split

Here's why we need to do this at this point after looking at the data:

📌 Is It a Big Problem If You Do train_test_split After One-Hot Encoding & Normalization?
No, it’s not the end of the world, but it’s not ideal either. Here’s why:

🚨 Why It’s Better to Split Before Preprocessing

✅ Prevents Data Leakage - If you apply Min-Max Scaling or Standardization before splitting, the test set will influence the transformation.
This means your test set is no longer truly "unseen" because the normalization used information from it.

✅ Avoids Issues with One-Hot Encoding - If a categorical value exists only in the test set, it won’t appear in the training data’s one-hot encoding, which may cause issues.
Solution: Use handle_unknown="ignore" in OneHotEncoder to avoid errors.





Also, our test_data doesn't have groundtruths


Since Your Test Data Doesn't Have Ground Truths (Kaggle-style), Here’s the Correct Approach
Because your test_data doesn't have labels (y_test), you should never use it for model tuning or selection. Instead:

1️⃣ Use Cross-Validation on X_normalized + y to tune hyperparameters.

2️⃣ Train the best model on the entire training set (after selecting the best hyperparameters).

3️⃣ Apply the same preprocessing to test_data (Min-Max scaling, encoding).

4️⃣ Use the trained model to make predictions on test_data.

5️⃣ Save the predictions for submission (e.g., Kaggle).

In [ ]:
# Split First (Best Practice)
X_train, X_test, y_train, y_test = train_test_split(train_data, y, test_size=0.2, random_state=42)


## Turn catagorial data into One-hot encoding

In [ ]:
from sklearn.preprocessing import OneHotEncoder


# Step 1: Identify categorical columns
categorical_cols = X_train.select_dtypes(include=['object']).columns  

# Step 2: Initialize OneHotEncoder (store it globally)
one_hot_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# 🚀 Step 3: Fit the encoder on X_train and Transform it
encoded_cats_train = one_hot_encoder.fit_transform(X_train[categorical_cols])  # ✅ Fit and Transform

# 🚀 Step 4: Convert Encoded Features to DataFrame
encoded_cats_train_df = pd.DataFrame(encoded_cats_train, columns=one_hot_encoder.get_feature_names_out(categorical_cols))

# 🚀 Step 5: Drop categorical columns and concatenate encoded features
X_train_encoded = X_train.drop(columns=categorical_cols).reset_index(drop=True)  # ✅ Store as `X_train_encoded`
X_train_encoded = pd.concat([X_train_encoded, encoded_cats_train_df], axis=1)

# ✅ SAVE the `one_hot_encoder` for later use!
print("✅ OneHotEncoder has been successfully fitted on X_train and saved!")


X_encoded = X_train_encoded


# step 6: Drop Sale price 
X_encoded = X_encoded.drop(columns=["SalePrice"])

In [ ]:
X_encoded.head()

In [ ]:
print(X_encoded.shape)

## Handling Null values

Investigating the number of Null values

#### Calude advised the below:

Based on the missing values distribution shown in your graph, I'll provide a strategic approach to handle these nulls. Let's break this down into groups based on the percentage of missing values and the likely nature of these features:

#### High Missing Values (>80%):

PoolQC, MiscFeature, Alley, Fence (80-100% missing)
These features are likely "naturally" missing - meaning the absence indicates the feature doesn't exist (e.g., no pool means no PoolQC). I recommend:


Create binary flags indicating presence/absence
Fill nulls with 'None' or a special category rather than dropping

#### Medium Missing Values (40-60%):

MasVnrType, FireplaceQu
These appear to be quality-related features. Suggested approach:
Consider creating binary presence flags
For quality ratings, use mode/median for filling based on related features (e.g., FireplaceQu could be filled based on house quality/year built)


#### Low Missing Values (<20%):

LotFrontage and various Garage/Basement features

For these:
LotFrontage: Use median values based on neighborhood or lot configuration
Garage features: Create a coordinated filling strategy since they're related
Basement features: Similarly, use a coordinated approach for all basement-related nulls



For implementing this, I'd suggest:
pythonCopy# Create binary flags for high-missing features
df['has_pool'] = df['PoolQC'].notna().astype(int)
df['has_fence'] = df['Fence'].notna().astype(int)

 Fill quality-related features
df['PoolQC'] = df['PoolQC'].fillna('None')
df['FireplaceQu'] = df['FireplaceQu'].fillna('None')

 Handle LotFrontage using neighborhood medians
lot_frontage_by_neighborhood = df.groupby('Neighborhood')['LotFrontage'].transform('median')
df['LotFrontage'] = df['LotFrontage'].fillna(lot_frontage_by_neighborhood)





### Alternative option from another notebook:

dropped the below options:

PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'LotFrontage'



In [ ]:
### check missing values


# Step 1: Count missing values per column
missing_values = X_encoded.isnull().sum()

# Step 2: Calculate percentage of missing values
missing_percentage = (missing_values / len(train_data)) * 100

# Step 3: Create a DataFrame for visualization
missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage': missing_percentage
})


In [ ]:
### check number of missing values


print(X_train.isnull().sum().max())  # Should be 0 if preprocessing was done correctly
print(X_test.isnull().sum().max())   #




print(train_data.isnull().sum().max())  # Should be 0 if preprocessing was done correctly
print(test_data.isnull().sum().max())   #

In [ ]:
# Show only columns with missing values
missing_df = missing_df[missing_df['Missing Values'] > 0]

# Step 4: Sort by the highest missing percentage
missing_df = missing_df.sort_values(by='Percentage', ascending=False)

# Display the missing values distribution 
print(missing_df)   # <-- Here, we simply print the DataFrame

# Visualization of Missing Values
plt.figure(figsize=(10, 6))
plt.barh(missing_df.index, missing_df['Percentage'], color='skyblue')
plt.xlabel('Percentage of Missing Values')
plt.ylabel('Columns')
plt.title('Missing Values Distribution')
plt.gca().invert_yaxis()  # Invert y-axis for better readability
plt.show()

In [ ]:
missing_df.head()

## Removing Null values - Median Imputation

We'll just go for replacing the null vlaues with the median results


In [ ]:
# Simple median imputation
X_encoded['LotFrontage'] = X_encoded['LotFrontage'].fillna(X_encoded['LotFrontage'].median())
X_encoded['GarageYrBlt'] = X_encoded['GarageYrBlt'].fillna(X_encoded['GarageYrBlt'].median())
X_encoded['MasVnrArea'] = X_encoded['MasVnrArea'].fillna(X_encoded['MasVnrArea'].median())

# Print the remaining null values to verify
print("\nRemaining null values after imputation:")
print(X_encoded[['LotFrontage', 'GarageYrBlt', 'MasVnrArea']].isnull().sum())

## Normalisation


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

# Initialize MinMaxScaler
scaler = MinMaxScaler()

# Fit and transform the entire dataset
X_normalized = scaler.fit_transform(X_encoded)

# Convert back to a DataFrame (preserves column names)
X_train_scaled = pd.DataFrame(X_normalized, columns=X_encoded.columns)

# Check the first few rows
print(X_train_scaled.head())



# Models

## Fit the Linear regression models

In [ ]:
# Initialize the Models
linear_model_minmax = LinearRegression()
ridge_model_minmax = Ridge(alpha=1.0)  # L2 Regularization (tunable hyperparameter)
lasso_model_minmax = Lasso(alpha=0.01)  

In [ ]:
# Train Each Model
linear_model_minmax.fit(X_train_scaled, y_train)
ridge_model_minmax.fit(X_train_scaled, y_train)
lasso_model_minmax.fit(X_train_scaled, y_train)



## Apply same pre-processing steps to the test set

In [ ]:
# Check if the OneHotEncoder has been fitted
try:
    print("Number of features in OneHotEncoder:", one_hot_encoder.get_feature_names_out())
except AttributeError:
    print("🚨 OneHotEncoder has NOT been fitted! Re-run the fitting step on X_train.")

In [ ]:
# Ensure SalePrice is Removed from X_test
X_test = X_test.drop(columns=["SalePrice"], errors="ignore")

# Use the Fitted Encoder to Transform X_test
encoded_cats_test = one_hot_encoder.transform(X_test[categorical_cols])  # ✅ Reuse the fitted encoder

# Convert to DataFrame
encoded_cats_test_df = pd.DataFrame(encoded_cats_test, columns=one_hot_encoder.get_feature_names_out(categorical_cols))

# Drop original categorical columns and concatenate new encoded features
X_test_encoded = X_test.drop(columns=categorical_cols).reset_index(drop=True)
X_test_encoded = pd.concat([X_test_encoded, encoded_cats_test_df], axis=1)

# Ensure column order and match train-test features
X_test_encoded['LotFrontage'] = X_test_encoded['LotFrontage'].fillna(X_train_encoded['LotFrontage'].median())
X_test_encoded['GarageYrBlt'] = X_test_encoded['GarageYrBlt'].fillna(X_train_encoded['GarageYrBlt'].median())
X_test_encoded['MasVnrArea'] = X_test_encoded['MasVnrArea'].fillna(X_train_encoded['MasVnrArea'].median())

# Verify missing values are removed
print("✅ Missing values after imputation in X_test_encoded:")
print(X_test_encoded[['LotFrontage', 'GarageYrBlt', 'MasVnrArea']].isnull().sum())  # Should print 0


# Apply Simple median imputation to X_test_encoded
X_test_encoded['LotFrontage'] = X_test_encoded['LotFrontage'].fillna(X_test_encoded['LotFrontage'].median())
X_test_encoded['GarageYrBlt'] = X_test_encoded['GarageYrBlt'].fillna(X_encoded['GarageYrBlt'].median())
X_test_encoded['MasVnrArea'] = X_test_encoded['MasVnrArea'].fillna(X_encoded['MasVnrArea'].median())

In [ ]:
# 🚀 Check for NaN values in X_test_encoded before scaling
print("\n✅ Remaining null values in X_test_encoded before scaling:", X_test_encoded.isnull().sum().sum())  # Should be 0

In [ ]:
X_test_scaled = scaler.transform(X_test_encoded)  # ✅ Use same scaler from earlier
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test_encoded.columns)  # Convert back to DataFrame

# 🚀 Final check
print("✅ X_test_scaled is now fully processed and ready for model predictions!")
print("\n✅ NaN values in X_test_scaled:", np.isnan(X_test_scaled_df).sum().sum())

## Assess the model - min_max scaling


In [ ]:
### before running the model, check that the shapes are consistent and there are no null values in either test or train 

print("✅ X_train_encoded shape:", X_train_encoded.shape)
print("✅ X_test_encoded shape:", X_test_encoded.shape)
print("✅ Scaler fitted on:", scaler.n_features_in_)



print("\n✅ NaN values in X_test_scaled:", np.isnan(X_test_scaled).sum().sum())
print("\n✅ NaN values in X_train_scaled:", np.isnan(X_train_scaled).sum().sum())

In [ ]:
# Get feature names the scaler was fitted on
scaler_features = set(scaler.feature_names_in_)

# Get current feature names
train_features = set(X_train_encoded.columns)

# Find the missing column
missing_features = scaler_features - train_features

print("🚨 Missing feature(s):", missing_features)

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

# Train Models
linear_model_minmax.fit(X_train_scaled, y_train)
ridge_model_minmax.fit(X_train_scaled, y_train)
lasso_model_minmax.fit(X_train_scaled, y_train)

# Make Predictions
y_pred_lin_minmax = linear_model_minmax.predict(X_test_scaled)
y_pred_ridge_minmax = ridge_model_minmax.predict(X_test_scaled)
y_pred_lasso_minmax = lasso_model_minmax.predict(X_test_scaled)

print("✅ Model predictions successful!")

# Evaluate Model Performance (Mean Squared Error)
mse_linear = mean_squared_error(y_test, y_pred_lin_minmax)
mse_ridge = mean_squared_error(y_test, y_pred_ridge_minmax)
mse_lasso = mean_squared_error(y_test, y_pred_lasso_minmax)

#  Print Results
print(f"Linear Regression MSE: {mse_linear:.4f}")
print(f"Ridge Regression MSE: {mse_ridge:.4f}")
print(f"Lasso Regression MSE: {mse_lasso:.4f}")


# Compute metrics
print("### MinMax Scaling ###")
for model_name, y_pred in [
    ("Linear (MinMax)", y_pred_lin_minmax),
    ("Ridge  (MinMax)", y_pred_ridge_minmax),
    ("Lasso  (MinMax)", y_pred_lasso_minmax),
]:
    print(f"{model_name} R²: {r2_score(y_test, y_pred):.3f} | MSE: {mean_squared_error(y_test, y_pred):.3f}")

In [ ]:
print("Feature Ranges in X_train_scaled:")
print("Min:", np.min(X_train_scaled, axis=0))
print("Max:", np.max(X_train_scaled, axis=0))

## Assess the model - standard scaling


#### Try the model with standard sclaing rather than 0-1 scaling

In [ ]:
scaler = StandardScaler()  # 🚀 Try Standard Scaling
X_train_scaled_standard = scaler.fit_transform(X_encoded)
X_test_scaled_standard = scaler.transform(X_test_encoded)

In [ ]:

print("\n✅ NaN values in X_test_scaled_standard:", np.isnan(X_test_scaled_standard).sum().sum())
print("\n✅ NaN values in X_train_scaled_standard:", np.isnan(X_train_scaled_standard).sum().sum())
print("\n✅ NaN values in X_encoded:", np.isnan(X_encoded).sum().sum())

In [ ]:


# Initialize the Models specifically for standard scaling
linear_model_std = LinearRegression()
ridge_model_std = Ridge(alpha=1.0)  # L2 Regularization (tunable hyperparameter) - standard scaled Ridge
lasso_model_std = Lasso(alpha=0.01)  


In [ ]:

# Train Models
linear_model_std.fit(X_train_scaled_standard, y_train)
ridge_model_std.fit(X_train_scaled_standard, y_train)
lasso_model_std.fit(X_train_scaled_standard, y_train)

# Make Predictions
y_pred_linear_stand = linear_model_std.predict(X_test_scaled_standard)
y_pred_ridger_stand = ridge_model_std.predict(X_test_scaled_standard)
y_pred_lassor_stand = lasso_model_std.predict(X_test_scaled_standard)

print("✅ Model predictions successful!")

# Evaluate Model Performance (Mean Squared Error)
mse_linear = mean_squared_error(y_test, y_pred_linear_stand)
mse_ridge = mean_squared_error(y_test, y_pred_ridger_stand)
mse_lasso = mean_squared_error(y_test, y_pred_lassor_stand)

# Metrics
print("### Standard Scaling ###")
for model_name, y_pred in [
    ("Linear (Std)", y_pred_linear_stand),
    ("Ridge  (Std)", y_pred_ridger_stand),
    ("Lasso  (Std)", y_pred_lassor_stand),
]:
    print(f"{model_name} R²: {r2_score(y_test, y_pred):.3f} | MSE: {mean_squared_error(y_test, y_pred):.3f}")

## Assess the model - choosing selected features

In [ ]:

print("\n✅ NaN values in X_test_scaled:", np.isnan(X_test_scaled).sum().sum())
print("\n✅ NaN values in X_train_scaled:", np.isnan(X_train_scaled).sum().sum())

In [ ]:
# Convert scaled arrays back to DataFrames using original feature names
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train_encoded.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test_encoded.columns)



new_train = X_train_scaled_df[['Id', 'ExterQual_Ex', 'ExterQual_Fa', 'ExterQual_Gd', 'ExterQual_TA', 'OverallQual', 'GrLivArea', 'GarageCars', 
                   'GarageArea']]
#------------------------------------------------------------------------------------------------
new_test = X_test_scaled_df[['Id', 'ExterQual_Ex', 'ExterQual_Fa', 'ExterQual_Gd', 'ExterQual_TA', 'OverallQual', 'GrLivArea', 'GarageCars', 
                   'GarageArea']]



 

In [ ]:

# Initialize the Models specifically for standard scaling
linear_model_selected = LinearRegression()
ridge_model_selected = Ridge(alpha=1.0)  # L2 Regularization (tunable hyperparameter) - standard scaled Ridge
lasso_model_selected = Lasso(alpha=0.01)  

In [ ]:
# Train Models
linear_model_selected.fit(new_train, y_train)
ridge_model_selected.fit(new_train, y_train)
lasso_model_selected.fit(new_train, y_train)

# Make Predictions
y_pred_linear_new = linear_model_selected.predict(new_test)  
y_pred_ridge_new = ridge_model_selected.predict(new_test)
y_pred_lasso_new = lasso_model_selected.predict(new_test)

# ✅ Use new_test for predictions


print("✅ Model predictions successful!")

# Evaluate Model Performance (Mean Squared Error)
# Now compare correctly
mse_linear = mean_squared_error(y_test, y_pred_linear_new)
mse_ridge = mean_squared_error(y_test, y_pred_ridge_new)
mse_lasso = mean_squared_error(y_test, y_pred_lasso_new)


# Evaluate
print("### Selected Features (MinMax) ###")
for model_name, y_pred in [
    ("Linear (Sel)", y_pred_linear_new),
    ("Ridge  (Sel)", y_pred_ridge_new),
    ("Lasso  (Sel)", y_pred_lasso_new),
]:
    print(f"{model_name} R²: {r2_score(y_test, y_pred):.3f} | MSE: {mean_squared_error(y_test, y_pred):.3f}")


## Run Gradient Descent

Now that X_train is correctly defined, we can run gradient descent to optimize the cost function.

In [ ]:
from sklearn.preprocessing import StandardScaler


# Convert data to NumPy arrays for efficient computation
X_train_np = np.array(X_train_scaled)
y_train_np = np.array(y_train).reshape(-1, 1)

# Initialize model parameters (weights) to zeros
theta = np.zeros((X_train_np.shape[1], 1))

# Set hyperparameters
learning_rate = 0.00001  # Step size for gradient descent
num_iterations = 20000  # Number of iterations
lambda_reg = 1  # Regularization strength (for Ridge regression)

# Number of training examples
m = X_train_np.shape[0]

# Store cost function history
cost_history = []

# Gradient Descent Algorithm
for i in range(num_iterations):
    # Compute predictions
    y_pred = np.dot(X_train_np, theta)

    # Compute the error
    error = y_pred - y_train_np

    # Compute the cost function (Mean Squared Error + Ridge Regularization)
    cost = (1 / (2 * m)) * np.sum(error ** 2) + (lambda_reg / (2 * m)) * np.sum(theta ** 2)
    cost_history.append(cost)

    # Compute the gradients
    gradients = (1 / m) * np.dot(X_train_np.T, error) + (lambda_reg / m) * theta

    # Update theta using gradient descent
    theta -= learning_rate * gradients

# Store final model parameters
final_theta = theta



# Display cost function history
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(range(num_iterations), cost_history, label="Cost Function")
plt.xlabel("Iterations")
plt.ylabel("Cost")
plt.title("Cost Function Convergence")
plt.legend()
plt.show()

# With this:
theta_df = pd.DataFrame(final_theta, 
                       index=X_train_scaled.columns,
                       columns=["Theta Values"])
print(theta_df)

In [ ]:
theta_df.shape

## Run Gadient Descent - Update the parameters

In [ ]:
# Flatten theta, currentl (302,1)/(n_features,1) needs to have dimensions (302,)/(n_features_,)
final_theta = final_theta.flatten()  # ensures shape (n_features,)


# Function to make predictions
def predict(X_train_scaled, theta):
    # Make sure X is in the same scaled and encoded format as training data
    return np.dot(X_train_scaled, theta)

# Make predictions on training set
y_grad_pred_train = predict(X_train_scaled, final_theta)

# For test set predictions (assuming you have a test set)
# First apply the same scaling and encoding as you did to training data
#X_test_scaled = scaler.transform(X_test)  # Use the same scaler as training
# Apply the same one-hot encoding transformation
# ... your one-hot encoding step here ...

# X_test_np = np.array(X_test_scaled)
y_grad_pred_test = predict(X_test_scaled, final_theta)

# Calculate R-squared score for training set
from sklearn.metrics import r2_score
r2_train = r2_score(y_train, y_grad_pred_train)
print(f"R-squared score on training set: {r2_train}")

# If you have a test set:
r2_test = r2_score(y_test, y_grad_pred_test)
print(f"R-squared score on test set: {r2_test}")

# You can also calculate MSE
mse_train = np.mean((y_train - y_grad_pred_train) ** 2)
print(f"Mean squared error on training set: {mse_train}")

In [ ]:
### save results of gradient descet predictions 

r2_gd_train = r2_score(y_train, y_grad_pred_train)
r2_gd_test  = r2_score(y_test,  y_grad_pred_test)
mse_gd_train = mean_squared_error(y_train, y_grad_pred_train)
mse_gd_test  = mean_squared_error(y_test,  y_grad_pred_test)

print(f"GD (MinMax) Train R²: {r2_gd_train:.3f}  |  Test R²: {r2_gd_test:.3f}")
print(f"GD (MinMax) Train MSE: {mse_gd_train:.3f} |  Test MSE: {mse_gd_test:.3f}")

## Model comparison

Compare the performance of Lasso regression, linear regression and ridge regression

Also compare their performance with one hot endocing vs label encoding



In [ ]:
# Suppose you have these predictions and the true labels already:
#   y_pred_linear_new, y_pred_ridge_new, y_pred_lasso_new
#   y_test
# And you want to gather them into a single results table.

results = []  # this will collect all of our metric dictionaries

# Example: selected features (MinMax)
for model_name, y_pred in [
    ("Linear (Sel)",  y_pred_linear_new),
    ("Ridge (Sel)",   y_pred_ridge_new),
    ("Lasso (Sel)",   y_pred_lasso_new),
]:
    # Compute metrics
    r2  = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    
    # Append to results list
    results.append({
        "Model": model_name,
        "Scaling": "Selected (MinMax)",  # or whatever label you want
        "R^2": r2,
        "MSE": mse
    })

# Example: standard scaling
for model_name, y_pred in [
    ("Linear (Std)",  y_pred_linear_stand),
    ("Ridge (Std)",   y_pred_ridger_stand),
    ("Lasso (Std)",   y_pred_lassor_stand),
]:
    r2  = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    
    results.append({
        "Model": model_name,
        "Scaling": "Standard",
        "R^2": r2,
        "MSE": mse
    })

# Example: min-max scaling
for model_name, y_pred in [
    ("Linear (MinMax)", y_pred_lin_minmax),
    ("Ridge (MinMax)",  y_pred_ridge_minmax),
    ("Lasso (MinMax)",  y_pred_lasso_minmax),
]:
    r2  = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    
    results.append({
        "Model": model_name,
        "Scaling": "MinMax",
        "R^2": r2,
        "MSE": mse
    })




# Gradient Descent example
results.append({
    "Model": "GD (MinMax)",
    "Scaling": "MinMax",
    "R^2": r2_gd_test,
    "MSE": mse_gd_test
})

# Convert the list of dicts into a DataFrame
df_results = pd.DataFrame(results)

# (Optional) round the values, style, or reorder columns
df_results = df_results.round(3)



# Display
display(df_results)


# Submission

### Check the test data has the same missing values as the training data

In [ ]:
#### Test data needs the same preprocessing that the training data went through


### Firstly, apply one hot encoding then see what Null values there are 

# Apply the fitted OneHotEncoder to test_data
encoded_cats_test = one_hot_encoder.transform(test_data[categorical_cols]) 

# Convert to DataFrame
encoded_cats_test_df = pd.DataFrame(encoded_cats_test, columns=one_hot_encoder.get_feature_names_out(categorical_cols))


# Drop original categorical columns and concatenate new encoded features
test_data_encoded = test_data.drop(columns=categorical_cols).reset_index(drop=True)
test_data_encoded = pd.concat([test_data_encoded, encoded_cats_test_df], axis=1)


# Step 1: Count missing values per column
missing_values_test = test_data_encoded.isnull().sum()

# Step 2: Calculate the percentage of missing values
missing_percentage_test = (missing_values_test / len(test_data_encoded)) * 100

# Step 3: Create a DataFrame for visualization
missing_df_test = pd.DataFrame({
    'Missing Values': missing_values_test,
    'Percentage': missing_percentage_test
})

# Step 4: Show only columns with missing values
missing_df_test = missing_df_test[missing_df_test["Missing Values"] > 0]

# Step 5: Sort by the highest missing percentage
missing_df_test = missing_df_test.sort_values(by='Percentage', ascending=False)

# Display missing values
if missing_df_test.empty:
    print("✅ No missing values in test_data_encoded!")
else:
    print("⚠️ Missing values detected in test_data_encoded:")
    print(missing_df)

In [ ]:

# Step 4: Sort by the highest missing percentage
missing_df_test = missing_df_test.sort_values(by='Percentage', ascending=False)

# Display the missing values distribution 
print(missing_df_test)   # <-- Here, we simply print the DataFrame



### Plot the missing test data

In [ ]:
# Function to create histograms for columns with missing values
def plot_histograms_for_columns(df, columns_with_missing=None):
    """
    Plot histograms for specified columns in a dataframe
    
    Parameters:
    df (pandas.DataFrame): The dataframe containing the data
    columns_with_missing (list): List of column names to plot histograms for
    """
    # If no columns specified, use all columns
    if columns_with_missing is None:
        columns_with_missing = df.columns.tolist()
    
    # Set up the figure style
    plt.style.use('seaborn-v0_8-darkgrid')
    
    # Plot each column in a separate figure for better visibility
    for column in columns_with_missing:
        # Skip if column doesn't exist
        if column not in df.columns:
            print(f"Column '{column}' not found in dataframe")
            continue
            
        # Create a figure
        plt.figure(figsize=(10, 6))
        
        # Get non-missing values for the column
        data = df[column].dropna()
        
        # Calculate the number of missing values
        missing_count = df[column].isna().sum()
        missing_percent = (missing_count / len(df)) * 100
        
        # Create the histogram
        sns.histplot(data, kde=True, bins=30)
        
        # Add title and labels
        plt.title(f'Histogram of {column} (Missing: {missing_count} values, {missing_percent:.2f}%)', fontsize=14)
        plt.xlabel(column, fontsize=12)
        plt.ylabel('Frequency', fontsize=12)
        
        # Add statistical information as text
        stats_text = (
            f"Statistics (non-missing values):\n"
            f"Count: {len(data)}\n"
            f"Mean: {data.mean():.2f}\n"
            f"Median: {data.median():.2f}\n"
            f"Std Dev: {data.std():.2f}\n"
            f"Min: {data.min():.2f}\n"
            f"Max: {data.max():.2f}"
        )
        
        plt.figtext(0.15, 0.70, stats_text, fontsize=10, 
                   bbox=dict(facecolor='white', alpha=0.8))
        
        plt.tight_layout()
        plt.show()

# List of columns with missing values based on your data
columns_with_missing = [
    'LotFrontage', 'GarageYrBlt', 'MasVnrArea', 'BsmtFullBath', 
    'BsmtHalfBath', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 
    'TotalBsmtSF', 'GarageCars', 'GarageArea'
]

# Plot histograms for all columns with missing values using your dataframe
plot_histograms_for_columns(test_data_encoded, columns_with_missing)

### Handle the Null test values

In [ ]:
# Calculate medians from your training data
numerical_cols = train_data.select_dtypes(include=['int64', 'float64']).columns
medians = train_data[numerical_cols].median()

# Loop through each numerical column and impute missing values directly
for col in numerical_cols:
    if col in test_data_encoded.columns and test_data_encoded[col].isna().any():
        # Replace NaN values with median from training data
        test_data_encoded[col].fillna(medians[col], inplace=True)

# Now test_data_encoded has all missing numerical values filled with training data medians

In [ ]:
# recheck the missing values


print("\n✅ NaN values in test_data_encoded:", np.isnan(test_data_encoded).sum().sum())

In [ ]:
#### Standard Scaler ####

# Apply the fitted StandardScaler to test data
test_final_scaled = scaler.transform(test_data_encoded)  # ✅ Use same scaler from earlier

# Convert back to DataFrame using correct column names from test_data_encoded
test_final_scaled_df = pd.DataFrame(test_final_scaled, columns=test_data_encoded.columns)

# 🚀 Final check
print("✅ test_final_scaled is now fully processed and ready for model predictions!")
print("\n✅ NaN values in test_final_scaled_df:", np.isnan(test_final_scaled_df).sum().sum())


In [ ]:
# Ensure 'SalePrice' is NOT in the training feature set
train_features_final = X_train_scaled.drop(columns=['SalePrice'], errors='ignore')

# Identify missing and extra columns in test data
missing_cols = set(train_features_final.columns) - set(test_final_scaled_df.columns)
extra_cols = set(test_final_scaled_df.columns) - set(train_features_final.columns)

print(f"Missing columns in test set: {missing_cols}")
print(f"Extra columns in test set: {extra_cols}")

In [ ]:

# Reorder columns to match training data
test_final_scaled_df = test_final_scaled_df[train_features_final.columns]


# Convert to NumPy array for prediction
test_final_scaled_np = np.array(test_final_scaled_df)



# Make Predictions using the trained Ridge model
y_pred_final = ridge_model_std.predict(test_final_scaled_np)
print("✅ Ridge Model predictions successful!")


# Ensure predictions are in the correct format
predictions_df = pd.DataFrame({
    "Id": test_data["Id"],  # Ensure ID is included
    "SalePrice": y_pred_final.ravel()  # Flatten in case it's a multi-dimensional array
})



# Define a writable output path
output_path = "/kaggle/working/submission.csv" 

# Save predictions to CSV
predictions_df.to_csv(output_path, index=False)

print(f"✅ Predictions saved to {output_path}")

In [67]:
predictions_df.head()

,Id,SalePrice
0,1461,120824.714177
1,1462,160671.787187
2,1463,188747.172764
3,1464,194483.251345
4,1465,213536.388521
